Splitting the dataset to train test and valifdation for MSVD 


In [6]:
import os
import random
import shutil
import glob
from collections import defaultdict

# ====== CONFIGURATION ======
video_dir = r"D:\major project\Video_captioning_code\msvd\YouTubeClips"
annotations_path = r"D:\major project\Video_captioning_code\captions\nepali_annotations.txt"
output_dir = r"D:\major project\Video_captioning_code\msvd_splitted"

train_count = 1200
val_count = 670
test_count = 100

# ====== STEP 1: LOAD ANNOTATIONS ======
annotations_dict = defaultdict(list)
with open(annotations_path, 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f):
        parts = line.strip().split(maxsplit=1)
        if len(parts) == 2:
            video_id, caption = parts
            annotations_dict[video_id].append(caption)
        else:
            print(f"⚠️ Skipped line {idx + 1}: {line.strip()}")

print(f"✅ Loaded {len(annotations_dict)} unique video IDs from annotation file.")

# ====== STEP 2: VALIDATE AND SPLIT IDS ======
all_video_ids = list(annotations_dict.keys())
total_available = len(all_video_ids)

assert total_available >= (train_count + val_count + test_count), \
    f"Only {total_available} videos available, but {train_count + val_count + test_count} needed."

random.seed(42)
random.shuffle(all_video_ids)

train_ids = all_video_ids[:train_count]
val_ids = all_video_ids[train_count:train_count + val_count]
test_ids = all_video_ids[train_count + val_count:train_count + val_count + test_count]

splits = {
    'train': train_ids,
    'val': val_ids,
    'test': test_ids
}

# ====== STEP 3: COPY VIDEOS & THEN WRITE CAPTIONS ======
for split_name, split_ids in splits.items():
    print(f"\n📝 Processing {split_name} split with {len(split_ids)} videos.")
    split_video_dir = os.path.join(output_dir, split_name, 'videos')
    os.makedirs(split_video_dir, exist_ok=True)

    copied_entries = []

    for idx, video_id in enumerate(split_ids, 1):
        pattern = os.path.join(video_dir, f"{video_id}*")
        matched_files = glob.glob(pattern)

        if matched_files:
            for src_path in matched_files:
                filename = os.path.basename(src_path)
                new_filename = f"{idx:05d}_{filename}"
                dst_path = os.path.join(split_video_dir, new_filename)
                shutil.copy2(src_path, dst_path)
                copied_entries.append((f"{idx:05d}", video_id))  # Track index + original ID
        else:
            print(f"⚠️ Warning: No video file found starting with ID: {video_id}")

    # Sort and write captions in the same order
    caption_output_path = os.path.join(output_dir, split_name, f"{split_name}.txt")
    with open(caption_output_path, 'w', encoding='utf-8') as f:
        for index_prefix, video_id in copied_entries:
            captions = annotations_dict.get(video_id, [])
            for caption in captions:
                f.write(f"{index_prefix}_{video_id}\t{caption}\n")

print("\n✅ All splits processed and aligned successfully.")


✅ Loaded 1970 unique video IDs from annotation file.

📝 Processing train split with 1200 videos.

📝 Processing val split with 670 videos.

📝 Processing test split with 100 videos.

✅ All splits processed and aligned successfully.


Verify if there is unique ids equal in captions and videos or not in each folders

In [7]:
import os

split_dir = r'D:\major project\Video_captioning_code\msvd_splitted'
splits = ['train', 'val', 'test']

for split in splits:
    print(f"\nChecking {split.upper()} folder")

    # Paths
    caption_file = os.path.join(split_dir, split, f"{split}.txt")
    video_folder = os.path.join(split_dir, split, 'videos')

    # 1. Unique video IDs in caption file
    caption_ids = set()
    with open(caption_file, 'r', encoding='utf-8') as f:
        for line in f:
            video_id = line.strip().split('\t')[0].strip()
            caption_ids.add(video_id)

    # 2. Unique video files in folder
    video_ids = set()
    for filename in os.listdir(video_folder):
        video_id, _ = os.path.splitext(filename)
        video_ids.add(video_id.strip())

    # 3. Compare sets
    only_in_captions = caption_ids - video_ids
    only_in_videos = video_ids - caption_ids

    print(f"Unique video IDs in captions: {len(caption_ids)}")
    print(f"Unique video files:          {len(video_ids)}")

    if only_in_captions:
        print(f"{len(only_in_captions)} video IDs in captions but no video file:")
        print(list(only_in_captions)[:10])
    else:
        print("All captioned video IDs have corresponding video files.")

    if only_in_videos:
        print(f"{len(only_in_videos)} video files not mentioned in captions:")
        print(list(only_in_videos)[:10])
    else:
        print("All video files have corresponding caption entries.")



Checking TRAIN folder
Unique video IDs in captions: 1200
Unique video files:          1200
All captioned video IDs have corresponding video files.
All video files have corresponding caption entries.

Checking VAL folder
Unique video IDs in captions: 670
Unique video files:          670
All captioned video IDs have corresponding video files.
All video files have corresponding caption entries.

Checking TEST folder
Unique video IDs in captions: 100
Unique video files:          100
All captioned video IDs have corresponding video files.
All video files have corresponding caption entries.


Translation using NLLB

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pathlib import Path
from tqdm.auto import tqdm                    # NEW ✨

# ----------------------- 1. CONFIG -----------------------
model_name   = "facebook/nllb-200-distilled-600M"
src_lang     = "eng_Latn"
tgt_lang     = "npi_Deva"
input_file   = Path("captions.txt")     # <- local file, adjust path
output_file  = Path("captions_ne.txt")
batch_size   = 32

# ----------------------- 2. SETUP ------------------------
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

tokenizer.src_lang = src_lang
forced_bos_id      = tokenizer.convert_tokens_to_ids(tgt_lang)

# ----------------------- 3. LOAD DATA --------------------
with input_file.open(encoding="utf-8") as f:
    captions = [line.strip() for line in f if line.strip()]

total_lines = len(captions)

# ----------------------- 4. TRANSLATE --------------------
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i : i + n]

nepali_captions = []
model.eval()

with torch.no_grad():
    for batch in tqdm(chunks(captions, batch_size),
                      total=(total_lines + batch_size - 1) // batch_size,
                      desc="Translating",
                      unit="batch"):
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(device)
        generated = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_id,
            max_length=30
        )
        nepali_captions.extend(
            tokenizer.batch_decode(generated, skip_special_tokens=True)
        )

# ----------------------- 5. SAVE OUTPUT ------------------
with output_file.open("w", encoding="utf-8") as f:
    for line in nepali_captions:
        f.write(line + "\n")

print(f"✅  Translated {total_lines} captions → {output_file.resolve()}")
